# Environment Setup and Authentication

In [1]:
import sys
import os

# Upgrade Hugging Face ecosystem safely
!{sys.executable} -m pip install --upgrade --quiet transformers accelerate huggingface_hub datasets

import torch
import math
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

# Hardware check
device_count = torch.cuda.device_count()
print(f"PyTorch {torch.__version__} | Detected {device_count} GPUs.")

# Authenticatation
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Authenticated via Kaggle Secrets.")
except Exception as e:
    HF_TOKEN = "" 
    login(token=HF_TOKEN)
    print("Authenticated via explicit string.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 98.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 86.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 34.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
PyTorch 2.10.0+cu128 | Detected 2 GPUs.
Authenticated via Kaggle Secrets.


# Evaluation Metrices

In [2]:
class IntrinsicEvaluator:
    def __init__(self, model, tokenizer, device="cuda:0"):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    def get_lengths(self, text: str):
        """Calculates text-intrinsic properties for normalization."""
        if not isinstance(text, str):
            text = str(text)
        num_chars = max(1, len(text))
        num_bytes = max(1, len(text.encode('utf-8')))
        num_words = max(1, len(text.strip().split()))
        return num_chars, num_bytes, num_words

    def evaluate_sequence(self, text: str):
        """Calculates raw NLL plus the raw counts needed to pool correctly across the
        whole corpus. bpb/bpc/bpw below are still returned for per-sentence inspection,
        but the CORPUS-LEVEL (headline) numbers must be computed by pooling nll and the
        *_count fields across all rows -- never by averaging these per-row ratios."""
        chars, bytes_count, words = self.get_lengths(text)
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs, labels=inputs["input_ids"])
            seq_len = inputs["input_ids"].shape[1]
            n_scored = seq_len - 1          # tokens actually predicted -- needed for token PPL
            neg_log_likelihood = outputs.loss.item() * n_scored

        bpb = neg_log_likelihood / (bytes_count * math.log(2))
        bpc = neg_log_likelihood / (chars * math.log(2))
        bpw = neg_log_likelihood / (words * math.log(2))

        return {
            "nll": neg_log_likelihood,
            "bpb": bpb, "bpc": bpc, "bpw": bpw,
            "n_scored": n_scored,
            "n_chars": chars, "n_bytes": bytes_count, "n_words": words,
        }
print("IntrinsicEvaluator class defined.")

IntrinsicEvaluator class defined.


# Data Loading and Preprocessing

In [3]:
diverse_sentences_path = "/kaggle/input/datasets/anon-owner-f/intrinsic-evaluation-dataset/diverse_sentences.csv"
sinhala_mixed_path = "/kaggle/input/datasets/anon-owner-f/intrinsic-evaluation-dataset/sinhala_mixed_dataset.csv"

try:
    diverse_df = pd.read_csv(diverse_sentences_path).dropna()
    mixed_df = pd.read_csv(sinhala_mixed_path).dropna()
    print(f"Loaded Diverse Sentences: {len(diverse_df)} parallel pairs.")
    print(f"Loaded Mixed Script: {len(mixed_df)} sentences.")
except Exception as e:
    print(f"Data loading failed. Check your file paths. Error: {e}")

Loaded Diverse Sentences: 500 parallel pairs.
Loaded Mixed Script: 500 sentences.


# Model & Tokenizer Initialization

In [4]:
model_id = "MBZUAI/LaMini-GPT-1.5B"
print(f"Loading {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model natively in fp16 across both GPUs
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype=torch.float16,
    token=HF_TOKEN,
    low_cpu_mem_usage=True
)
model.eval()

# Initialize our custom evaluator
evaluator = IntrinsicEvaluator(model, tokenizer)

print(f"Model loaded.")
print(f"Precision: {model.dtype} | Device Map: {model.hf_device_map}")

Loading MBZUAI/LaMini-GPT-1.5B...


config.json:   0%|          | 0.00/984 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/788 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/581 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: MBZUAI/LaMini-GPT-1.5B
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...47}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded.
Precision: torch.float16 | Device Map: {'transformer.wte': 0, 'lm_head': 0, 'transformer.wpe': 0, 'transformer.drop': 0, 'transformer.h.0': 0, 'transformer.h.1': 0, 'transformer.h.2': 0, 'transformer.h.3': 0, 'transformer.h.4': 0, 'transformer.h.5': 0, 'transformer.h.6': 0, 'transformer.h.7': 0, 'transformer.h.8': 0, 'transformer.h.9': 0, 'transformer.h.10': 0, 'transformer.h.11': 0, 'transformer.h.12': 0, 'transformer.h.13': 0, 'transformer.h.14': 0, 'transformer.h.15': 0, 'transformer.h.16': 0, 'transformer.h.17': 0, 'transformer.h.18': 0, 'transformer.h.19': 0, 'transformer.h.20': 0, 'transformer.h.21': 0, 'transformer.h.22': 1, 'transformer.h.23': 1, 'transformer.h.24': 1, 'transformer.h.25': 1, 'transformer.h.26': 1, 'transformer.h.27': 1, 'transformer.h.28': 1, 'transformer.h.29': 1, 'transformer.h.30': 1, 'transformer.h.31': 1, 'transformer.h.32': 1, 'transformer.h.33': 1, 'transformer.h.34': 1, 'transformer.h.35': 1, 'transformer.h.36': 1, 'transformer.h.37': 1, '

# Pilot Run

In [5]:
import csv

pilot_output_path = 'pilot_intrinsic_results.csv'
print("Starting Pilot Run (First 20 items)...")

with open(pilot_output_path, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow([
        'item_index',
        'unicode_nll', 'unicode_bpb', 'unicode_bpc', 'unicode_bpw',
        'unicode_nscored', 'unicode_nchars', 'unicode_nbytes', 'unicode_nwords',
        'romanized_nll', 'romanized_bpb', 'romanized_bpc', 'romanized_bpw',
        'romanized_nscored', 'romanized_nchars', 'romanized_nbytes', 'romanized_nwords'
    ])

    pilot_sample = diverse_df.head(20)

    for index, row in pilot_sample.iterrows():
        uni_metrics = evaluator.evaluate_sequence(row['sinhala_unicode'])
        rom_metrics = evaluator.evaluate_sequence(row['sinhala_romanized'])

        writer.writerow([
            index,
            uni_metrics['nll'], uni_metrics['bpb'], uni_metrics['bpc'], uni_metrics['bpw'],
            uni_metrics['n_scored'], uni_metrics['n_chars'], uni_metrics['n_bytes'], uni_metrics['n_words'],
            rom_metrics['nll'], rom_metrics['bpb'], rom_metrics['bpc'], rom_metrics['bpw'],
            rom_metrics['n_scored'], rom_metrics['n_chars'], rom_metrics['n_bytes'], rom_metrics['n_words']
        ])

        if (index + 1) % 5 == 0:
            print(f"Processed {index + 1}/20 parallel pairs...")

print(f"\n Pilot run complete. Results saved to {pilot_output_path}")
preview_df = pd.read_csv(pilot_output_path)
print("\nFirst 3 rows of calculated metrics:")
display(preview_df.head(3))

Starting Pilot Run (First 20 items)...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Processed 5/20 parallel pairs...
Processed 10/20 parallel pairs...
Processed 15/20 parallel pairs...
Processed 20/20 parallel pairs...

 Pilot run complete. Results saved to pilot_intrinsic_results.csv

First 3 rows of calculated metrics:


,item_index,unicode_nll,unicode_bpb,unicode_bpc,unicode_bpw,unicode_nscored,unicode_nchars,unicode_nbytes,unicode_nwords,romanized_nll,romanized_bpb,romanized_bpc,romanized_bpw,romanized_nscored,romanized_nchars,romanized_nbytes,romanized_nwords
0,0,127.260145,2.353815,6.119919,30.599597,77,30,78,6,99.609683,3.781747,3.781747,23.951066,15,38,38,6
1,1,100.565774,2.198269,6.045239,36.271436,65,24,66,4,71.022508,3.659422,3.659422,25.615955,11,28,28,4
2,2,121.794879,2.117023,5.668157,29.285478,82,31,83,6,95.574778,3.830146,3.830146,22.980876,14,36,36,6


# Full Intrinsic Evaluation

In [6]:
import csv
import pandas as pd
from tqdm import tqdm

full_diverse_output = 'lamini_gpt_1_5b_diverse_intrinsic.csv'
print(f"Starting full parallel evaluation on {len(diverse_df)} items...")

with open(full_diverse_output, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow([
        'item_index',
        'unicode_nll', 'unicode_bpb', 'unicode_bpc', 'unicode_bpw',
        'unicode_nscored', 'unicode_nchars', 'unicode_nbytes', 'unicode_nwords',
        'romanized_nll', 'romanized_bpb', 'romanized_bpc', 'romanized_bpw',
        'romanized_nscored', 'romanized_nchars', 'romanized_nbytes', 'romanized_nwords'
    ])

    for index, row in tqdm(diverse_df.iterrows(), total=len(diverse_df), desc="Processing Parallel Pairs"):
        uni_metrics = evaluator.evaluate_sequence(row['sinhala_unicode'])
        rom_metrics = evaluator.evaluate_sequence(row['sinhala_romanized'])

        writer.writerow([
            index,
            uni_metrics['nll'], uni_metrics['bpb'], uni_metrics['bpc'], uni_metrics['bpw'],
            uni_metrics['n_scored'], uni_metrics['n_chars'], uni_metrics['n_bytes'], uni_metrics['n_words'],
            rom_metrics['nll'], rom_metrics['bpb'], rom_metrics['bpc'], rom_metrics['bpw'],
            rom_metrics['n_scored'], rom_metrics['n_chars'], rom_metrics['n_bytes'], rom_metrics['n_words']
        ])

print(f"Diverse evaluation complete. Saved to {full_diverse_output}")

full_mixed_output = 'lamini_gpt_1_5b_mixed_intrinsic.csv'
mixed_text_col = 'text' if 'text' in mixed_df.columns else mixed_df.columns[0]
print(f"\nStarting full mixed-script evaluation on {len(mixed_df)} items (column: '{mixed_text_col}')...")

with open(full_mixed_output, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow([
        'item_index', 'mixed_nll', 'mixed_bpb', 'mixed_bpc', 'mixed_bpw',
        'mixed_nscored', 'mixed_nchars', 'mixed_nbytes', 'mixed_nwords'
    ])

    for index, row in tqdm(mixed_df.iterrows(), total=len(mixed_df), desc="Processing Mixed Script"):
        mix_metrics = evaluator.evaluate_sequence(row[mixed_text_col])
        writer.writerow([
            index,
            mix_metrics['nll'], mix_metrics['bpb'], mix_metrics['bpc'], mix_metrics['bpw'],
            mix_metrics['n_scored'], mix_metrics['n_chars'], mix_metrics['n_bytes'], mix_metrics['n_words']
        ])

print(f"Mixed-script evaluation complete. Saved to {full_mixed_output}")


# --- Corrected aggregation: pooled (corpus-level) BPB/BPC/BPW + token-level Perplexity ---
def calculate_full_aggregates(csv_path, model_label="LaMini-GPT-1.5B"):
    df = pd.read_csv(csv_path)
    ln2 = math.log(2)

    def pooled_stats(prefix):
        nll_sum = df[f'{prefix}_nll'].sum()
        return {
            'bpb': nll_sum / (df[f'{prefix}_nbytes'].sum() * ln2),
            'bpc': nll_sum / (df[f'{prefix}_nchars'].sum() * ln2),
            'bpw': nll_sum / (df[f'{prefix}_nwords'].sum() * ln2),
            # this is the number to compare against the paper's Table:
            'ppl_token': math.exp(nll_sum / df[f'{prefix}_nscored'].sum()),
        }

    uni = pooled_stats('unicode')
    rom = pooled_stats('romanized')

    results_df = pd.DataFrame({
        'Metric': ['Bits-Per-Byte (BPB)', 'Bits-Per-Character (BPC)', 'Bits-Per-Word (BPW)',
                   'Token-Level Perplexity'],
        'Unicode (Native)': [uni['bpb'], uni['bpc'], uni['bpw'], uni['ppl_token']],
        'Romanized (Singlish)': [rom['bpb'], rom['bpc'], rom['bpw'], rom['ppl_token']],
    })
    results_df['Degradation Factor'] = results_df['Romanized (Singlish)'] / results_df['Unicode (Native)']

    print("==========================================================")
    print(f"      {model_label} INTRINSIC EVALUATION SUMMARY (corpus-pooled)")
    print("==========================================================")
    print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("==========================================================")

    styled = results_df.style.format({
        'Unicode (Native)': '{:.4f}',
        'Romanized (Singlish)': '{:.4f}',
        'Degradation Factor': '{:.2f}×'
    }).set_caption(f"{model_label} — Unicode vs Romanized Sinhala (corpus-pooled)").background_gradient(
        subset=['Degradation Factor'], cmap='Reds'
    )
    display(styled)

    return results_df

full_summary = calculate_full_aggregates(full_diverse_output)

Starting full parallel evaluation on 500 items...




Processing Parallel Pairs:   0%|          | 0/500 [00:00<?, ?it/s]

Processing Parallel Pairs:   0%|          | 1/500 [00:00<01:14,  6.71it/s]

Processing Parallel Pairs:   0%|          | 2/500 [00:00<01:07,  7.39it/s]

Processing Parallel Pairs:   1%|          | 3/500 [00:00<01:03,  7.85it/s]

Processing Parallel Pairs:   1%|          | 4/500 [00:00<01:02,  7.96it/s]

Processing Parallel Pairs:   1%|          | 5/500 [00:00<01:00,  8.15it/s]

Processing Parallel Pairs:   1%|          | 6/500 [00:00<00:59,  8.35it/s]

Processing Parallel Pairs:   1%|▏         | 7/500 [00:00<00:59,  8.27it/s]

Processing Parallel Pairs:   2%|▏         | 8/500 [00:00<00:58,  8.41it/s]

Processing Parallel Pairs:   2%|▏         | 10/500 [00:01<00:55,  8.78it/s]

Processing Parallel Pairs:   2%|▏         | 11/500 [00:01<00:59,  8.27it/s]

Processing Parallel Pairs:   2%|▏         | 12/500 [00:01<01:00,  8.10it/s]

Processing Parallel Pairs:   3%|▎         | 13/500 [00:01<01:01,  7.88it/s]

Processing Par

Diverse evaluation complete. Saved to lamini_gpt_1_5b_diverse_intrinsic.csv

Starting full mixed-script evaluation on 500 items (column: 'text')...


Processing Mixed Script: 100%|██████████| 500/500 [00:23<00:00, 21.32it/s]

Mixed-script evaluation complete. Saved to lamini_gpt_1_5b_mixed_intrinsic.csv
      LaMini-GPT-1.5B INTRINSIC EVALUATION SUMMARY (corpus-pooled)
                  Metric  Unicode (Native)  Romanized (Singlish)  Degradation Factor
     Bits-Per-Byte (BPB)            2.3952                3.8272              1.5979
Bits-Per-Character (BPC)            6.3496                3.8379              0.6044
     Bits-Per-Word (BPW)           34.0555               23.6204              0.6936
  Token-Level Perplexity            5.4089             1322.0094            244.4125


,Metric,Unicode (Native),Romanized (Singlish),Degradation Factor
0,Bits-Per-Byte (BPB),2.3952,3.8272,1.60×
1,Bits-Per-Character (BPC),6.3496,3.8379,0.60×
2,Bits-Per-Word (BPW),34.0555,23.6204,0.69×
3,Token-Level Perplexity,5.4089,1322.0094,244.41×


In [7]:
def calculate_mixed_aggregates(csv_path, model_label="LaMini-GPT-1.5B"):
    df = pd.read_csv(csv_path)
    ln2 = math.log(2)
    nll_sum = df['mixed_nll'].sum()

    pooled_bpb = nll_sum / (df['mixed_nbytes'].sum() * ln2)
    pooled_bpc = nll_sum / (df['mixed_nchars'].sum() * ln2)
    pooled_bpw = nll_sum / (df['mixed_nwords'].sum() * ln2)
    pooled_ppl = math.exp(nll_sum / df['mixed_nscored'].sum())

    # per-sentence values, used ONLY for the dispersion column below -- not the headline numbers above
    per_sentence_ppl = (df['mixed_nll'] / df['mixed_nscored']).apply(math.exp)

    results_df = pd.DataFrame({
        'Metric': ['Bits-Per-Byte (BPB)', 'Bits-Per-Character (BPC)', 'Bits-Per-Word (BPW)',
                   'Token-Level Perplexity'],
        'Mixed-Script (corpus-pooled)': [pooled_bpb, pooled_bpc, pooled_bpw, pooled_ppl],
        'Per-sentence Std Dev': [df['mixed_bpb'].std(), df['mixed_bpc'].std(),
                                 df['mixed_bpw'].std(), per_sentence_ppl.std()],
    })

    print("==========================================================")
    print(f"      {model_label} MIXED-SCRIPT INTRINSIC SUMMARY")
    print(f"      ({len(df)} sentences)")
    print("==========================================================")
    print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("==========================================================")

    styled = results_df.style.format({
        'Mixed-Script (corpus-pooled)': '{:.4f}',
        'Per-sentence Std Dev': '{:.4f}'
    }).set_caption(f"{model_label} — Mixed-Script Sinhala")
    display(styled)

    return results_df

mixed_summary = calculate_mixed_aggregates(full_mixed_output)

      LaMini-GPT-1.5B MIXED-SCRIPT INTRINSIC SUMMARY
      (500 sentences)
                  Metric  Mixed-Script (corpus-pooled)  Per-sentence Std Dev
     Bits-Per-Byte (BPB)                        2.7876                0.4174
Bits-Per-Character (BPC)                        5.4948                0.6571
     Bits-Per-Word (BPW)                       33.3869                5.4594
  Token-Level Perplexity                        9.4582               58.5730


,Metric,Mixed-Script (corpus-pooled),Per-sentence Std Dev
0,Bits-Per-Byte (BPB),2.7876,0.4174
1,Bits-Per-Character (BPC),5.4948,0.6571
2,Bits-Per-Word (BPW),33.3869,5.4594
3,Token-Level Perplexity,9.4582,58.5730
